In [1]:
import os
import gc

import numpy as np
import pandas as pd

from glob import glob
from tqdm import tqdm

import matplotlib.pyplot as plt
from joblib import Parallel, delayed

In [2]:
final_ret_df = pd.read_pickle("DB/ret_w_delret.pkl")
not_na_mask = final_ret_df.notna()
first_valid_mask = not_na_mask & (not_na_mask.cumsum() == 1)
final_ret_df[first_valid_mask] = 0.0
cls = (final_ret_df + 1).cumprod()

vol = pd.read_pickle('DB/vol.pkl')
cap = pd.read_pickle('DB/cap.pkl')
raw_prc = pd.read_pickle('DB/raw_prc.pkl')

In [3]:
cls = cls.ffill()[~vol.isna()].copy()
cap = cap.ffill()[~vol.isna()].copy()
raw_prc = raw_prc.ffill()[~vol.isna()].copy()

cls = cls[cls.index>='2000-12-19'].dropna(how='all', axis=1)
vol = vol[vol.index>='2000-12-19'].dropna(how='all', axis=1)
cap = cap[cap.index>='2000-12-19'].dropna(how='all', axis=1)
raw_prc = raw_prc[raw_prc.index>='2000-12-19'].dropna(how='all', axis=1)

cls.index = pd.to_datetime(cls.index).strftime('%Y-%m-%d')
vol.index = pd.to_datetime(vol.index).strftime('%Y-%m-%d')
cap.index = pd.to_datetime(cap.index).strftime('%Y-%m-%d')
raw_prc.index = pd.to_datetime(raw_prc.index).strftime('%Y-%m-%d')

In [4]:
rebalnace_date = pd.read_csv("DB/rebalance_date.csv",index_col=0)['0']

In [ ]:
market_df =[]
for cap_threshold in [100,90,80,70,60,50]:

    ew_market = []
    ew_value = 1000

    for date in rebalnace_date[:]:
        temp = cls[date:].iloc[1:22].copy()
        temp = temp[temp.iloc[0].dropna().index].copy()
        temp = temp.ffill()

        # Cap limit , if cap_threshold == 100 Full universe, if 0 Empty universe
        cap_val = cap[cap.index == date].dropna(axis=1).T[date]
        cap_min = np.percentile(cap_val, 100 - cap_threshold)
        cap_val = cap_val[cap_val >= cap_min]
        cap_filter = list(cap_val.index)

        temp = temp[temp.columns[temp.columns.isin(cap_filter)]].copy()

        temp = (temp/temp.iloc[0]).mean(axis=1)
        temp = temp * ew_value 

        if len(ew_market) != len(rebalnace_date) -1 :
            ew_value = temp.iloc[-1]
            ew_market.append(temp.iloc[:-1])
        else:
            ew_value = temp.iloc[-1]
            temp.iloc[-1] = ew_value
            ew_market.append(temp)

    market = pd.DataFrame(pd.concat(ew_market))
    market.loc['2001-01-01'] = 1000
    market.columns = ['EW_CAP' + str(cap_threshold)]
    market.index = pd.to_datetime(market.index)
    market.sort_index(inplace=True)
    market_df.append(market)
    
pd.concat(market_df,axis=1).to_csv('DB/market_portfolio.csv')

In [ ]:
def cal_backtest(name,seed,cap_limit):
    
    cap_threshold = cap_limit
    base_path = f'./backtest_vff/{name}/{seed}/cap_limit{cap_threshold}'

    if not os.path.exists(base_path):
        os.makedirs(base_path)

    if not os.path.exists(base_path + '/size_breakdown'):
        os.makedirs(base_path + '/size_breakdown')

    sorting = 'normal'

    if name == 'STR':
        sorting = 'reverse'

    ports = {}
    value_weights = []
    cap_sec_info = []

    for i in range(10):
        ports[i] = []

    for date in tqdm(rebalnace_date):

        pred_df = pd.read_csv(r"monthly_prediction" + "/" + name + "/" + date + ".csv")
        pred_df = pred_df[['PERMNO', seed]]
        pred_df.columns = ['PERMNO','PRED']

        prc_seg = cls.loc[date:].iloc[1:22].copy()
        tradablestock_filter = list(prc_seg.iloc[0].dropna().index)
        pred_df = pred_df[pred_df['PERMNO'].isin(tradablestock_filter)].copy()

        # # 5$ filter raw prc
        # microstock_filter = list(raw_prc.loc[date][raw_prc.loc[date] >= 5].index)
        # pred_df = pred_df[pred_df['PERMNO'].isin(microstock_filter)].copy()

        # Cap limit , if cap_threshold == 100 Full universe, if 0 Empty universe
        cap_val = cap[cap.index == date].dropna(axis=1).T[date]
        cap_min = np.percentile(cap_val, 100 - cap_threshold)
        cap_val = cap_val[cap_val >= cap_min]
        cap_filter = list(cap_val.index)
        pred_df = pred_df[pred_df['PERMNO'].isin(cap_filter)].copy()

        # Cap Cap Proportion calculation
        whole_market = cap[cap.index == date].T[date].dropna().copy()
        cap_sector = pd.qcut(whole_market, 10, labels=False)
        cap_sec_info.append(cap_sector)
        value_weights.append(cap_val)

        if sorting == 'normal':
            pred_val = pred_df.set_index('PERMNO').sort_values('PRED',ascending=False)['PRED']
        else: # STR needs reverse
            pred_val = pred_df.set_index('PERMNO').sort_values('PRED',ascending=True)['PRED']
        pred_val.index.name = None

        # --------------
        prc_seg = prc_seg[pred_val.index].copy()
        rt_df = (prc_seg/prc_seg.iloc[0]).ffill()

        splits = np.array_split(list(pred_val.index), 10)

        for i in range(10):
            asset_lst = list(splits[i])
            port_rt = rt_df[asset_lst]
            ports[i].append(port_rt)

    equal_weight_port = {}
    value_weight_port = {}

    turnover_val_weight = {}
    turnover_equal_weight = {}
    turnover_return = {}

    equal_weight_decile = {}
    value_weight_decile = {}

    for i in range(10):
        
        equal_weight_port[i] = []
        value_weight_port[i] = []

        turnover_val_weight[i] = []
        turnover_equal_weight[i] = []
        turnover_return[i] = []

        equal_weight_decile[i] = []
        value_weight_decile[i] = []

        equal_weight_value = 1000
        value_weight_value = 1000

        for j in range(len(ports[i])): # j represents date

            if j >= 1 :
                equal_weight_port[i][-1] = equal_weight_port[i][-1].iloc[:-1]
                value_weight_port[i][-1] = value_weight_port[i][-1].iloc[:-1]
                
            asset_cnt = ports[i][j].shape[1]

            init_cap = value_weights[j].T[ports[i][j].columns]
            
            equal_weight = init_cap/init_cap * 1/asset_cnt 
            value_weight = init_cap/(init_cap.sum())

            val_cap_sec = pd.concat([value_weight,cap_sec_info[j]],axis=1).dropna(axis=0).copy()
            val_cap_sec.columns = ['weight','cap']
            value_weight_decile[i].append(val_cap_sec.groupby('cap').sum())

            eq_cap_sec = pd.concat([equal_weight,cap_sec_info[j]],axis=1).dropna(axis=0).copy()
            eq_cap_sec.columns = ['weight','cap']
            equal_weight_decile[i].append(eq_cap_sec.groupby('cap').sum())

            turnover_val_weight[i].append(value_weight)
            turnover_equal_weight[i].append(equal_weight)
            turnover_return[i].append(ports[i][j].ffill())

            # equal_weight_total_init_cost = equal_weight_value
            # value_weight_total_init_cost = value_weight_value

            # equal_weight_price_move = (equal_weight_value * equal_weight * ports[i][j].ffill()).sum(axis=1) - equal_weight_total_init_cost
            # value_weight_price_move = (value_weight_value * value_weight * ports[i][j].ffill()).sum(axis=1) - value_weight_total_init_cost

            # equal_total_exit_cost = equal_weight_price_move.iloc[-1]
            # value_total_exit_cost = value_weight_price_move.iloc[-1]

            # equal_weight_price_move.iloc[-1] -= equal_total_exit_cost
            # value_weight_price_move.iloc[-1] -= value_total_exit_cost

            equal_weight_price_move = (equal_weight_value * equal_weight * ports[i][j].ffill()).sum(axis=1)
            value_weight_price_move = (value_weight_value * value_weight * ports[i][j].ffill()).sum(axis=1)

            equal_weight_value = equal_weight_price_move.iloc[-1]
            value_weight_value = value_weight_price_move.iloc[-1]

            equal_weight_port[i].append(equal_weight_price_move)
            value_weight_port[i].append(value_weight_price_move)

    for key in equal_weight_port.keys():
        equal_weight_port[key] = pd.concat(equal_weight_port[key])
        value_weight_port[key] = pd.concat(value_weight_port[key])

    for i in range(10):
        # higher the bigger 
        value_decile_df = pd.concat(value_weight_decile[i],axis=1).T
        value_decile_df.index = rebalnace_date
        value_decile_df.index.name = None
        value_decile_df.columns.name = None
        value_decile_df.to_csv(base_path + '/size_breakdown/value_'+str(i)+'.csv')

        equal_decile_df = pd.concat(equal_weight_decile[i],axis=1).T
        equal_decile_df.index = rebalnace_date
        equal_decile_df.index.name = None
        equal_decile_df.columns.name = None
        equal_decile_df.to_csv(base_path + '/size_breakdown/equal_'+str(i)+'.csv')
    
    # cal turnover

    equal_turnover = {}
    value_turnover = {}
    
    for key in equal_weight_port.keys():

        equal_weight_change_df = pd.concat(turnover_equal_weight[key],axis=1).sort_index().T
        value_weight_change_df = pd.concat(turnover_val_weight[key],axis=1).sort_index().T

        date_lst = value_weight_change_df.index[1:]

        value_turnover[key] = []
        equal_turnover[key] = []

        for j in range(0,len(equal_weight_change_df) - 1):
            
            r_t = (turnover_return[key][j].iloc[-1]/turnover_return[key][j].iloc[0] - 1)
            vw_t0 = value_weight_change_df.iloc[j].dropna()
            ew_t0 = equal_weight_change_df.iloc[j].dropna()

            vw_t1 = value_weight_change_df.iloc[j+1].dropna()
            ew_t1 = equal_weight_change_df.iloc[j+1].dropna()

            date = vw_t1.name

            vw_table =pd.concat([vw_t0,vw_t1],axis=1)
            
            intersect_vw_table = vw_table[vw_table.index.isin(r_t.index)]
            new_vw_table = vw_table[~vw_table.index.isin(r_t.index)]

            ew_table =pd.concat([ew_t0,ew_t1],axis=1)
            
            intersect_ew_table = ew_table[ew_table.index.isin(r_t.index)]
            new_ew_table = ew_table[~ew_table.index.isin(r_t.index)]

            vw_turnover = np.abs(intersect_vw_table.fillna(0)[date] - ( (vw_t0 * (r_t + 1))  * 1/ (1 + (r_t * vw_t0).sum()) )).sum() + np.abs(new_vw_table.fillna(0)[date]).sum()
            ew_turnover = np.abs(intersect_ew_table.fillna(0)[date] - (  (ew_t0 * (r_t + 1))  * 1/ (1 + (r_t * ew_t0).sum()) )).sum() + np.abs(new_ew_table.fillna(0)[date]).sum()

            value_turnover[key].append(vw_turnover)
            equal_turnover[key].append(ew_turnover)
            
    gc.collect()

    # portfolio
    equal_port_df = pd.DataFrame(equal_weight_port)
    value_port_df = pd.DataFrame(value_weight_port)

    value_port_df = pd.DataFrame(value_weight_port)
    value_port_df = value_port_df.T
    value_port_df['2001-01-01'] = 1000
    value_port_df = value_port_df.T
    value_port_df.sort_index(inplace=True)

    value_port_df.to_csv(base_path + '/value_port.csv')

    equal_port_df = pd.DataFrame(equal_weight_port)
    equal_port_df = equal_port_df.T
    equal_port_df['2001-01-01'] = 1000
    equal_port_df = equal_port_df.T
    equal_port_df.sort_index(inplace=True)

    equal_port_df.to_csv(base_path + '/equal_port.csv')

    # turn over

    value_turnover_df = pd.DataFrame(value_turnover)
    value_turnover_df.index = date_lst

    value_turnover_df.to_csv(base_path + '/value_turnover.csv')

    equal_turnover_df = pd.DataFrame(equal_turnover)
    equal_turnover_df.index = date_lst

    equal_turnover_df.to_csv(base_path + '/equal_turnover.csv')

In [13]:
param_set = []

model_name = ['b32_30d_enc2',
            'gray']

for name in model_name:
    for seed in ['avg']: # 'seed14','seed51','seed60','seed71','seed92',
        for cap_lim in [100,90,80,70,60,50]:
            param_set.append([name,seed,cap_lim])

robust = ['b16_30d_enc2',
        'b32_20d_enc2',
        'b32_25d_enc2',
        'b32_30d_enc4',
        'b32_30d_enc6',]

for name in robust:
    for seed in ['avg']: # 'seed14','seed51','seed60','seed71','seed92',
        for cap_lim in [100]:
            param_set.append([name,seed,cap_lim])


for name in ['MOM','STR']:
    for seed in ['avg']:
            for cap_lim in [100,90,80,70,60,50]:
                param_set.append([name,seed,cap_lim])

# param_set = list(set(param_set))

param_set = [list(array) for array in list(pd.DataFrame(param_set).drop_duplicates().values)]

In [19]:
result = Parallel(n_jobs=-1, verbose=-1)(delayed(cal_backtest)(*params) for params in param_set)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done  22 out of  29 | elapsed:  1.3min remaining:   25.5s
[Parallel(n_jobs=-1)]: Done  29 out of  29 | elapsed:  1.4min finished
